In [ ]:
import numpy as np
import pandas as pd

DATA_DIR = "/workspace/data"
responses = pd.read_csv(f"{DATA_DIR}/responses.csv")
customers = pd.read_csv(f"{DATA_DIR}/customers.csv")
targets = pd.read_csv(f"{DATA_DIR}/targets.csv")
responses["response_date"] = pd.to_datetime(responses["response_date"])

WINDOW_DAYS = 90
DECAY_DAYS = 30
CONVERGENCE = 1e-9
latest = responses["response_date"].max()
cutoff = latest - pd.Timedelta(days=WINDOW_DAYS)


def weighted_median(values, weights):
    pairs = sorted(zip(values, weights), key=lambda x: x[0])
    total = sum(w for _, w in pairs)
    if total == 0:
        return float("nan")
    cum = 0.0
    half = total / 2.0
    for v, w in pairs:
        cum += w
        if cum >= half:
            return float(v)
    return float(pairs[-1][0])


def segment_metrics(segment):
    seg_customers = customers[customers["segment"] == segment].copy()
    complete = seg_customers.dropna(subset=["age_band", "region"]).copy()
    keep_ids = set(complete["customer_id"])

    q = responses[
        (responses["flag"] == "verified")
        & (responses["survey_type"] == "relationship")
        & (responses["response_date"] >= cutoff)
        & (responses["score"] != 7)
        & (responses["customer_id"].isin(keep_ids))
    ].copy()
    q["delta_days"] = (latest - q["response_date"]).dt.days
    q["w_resp"] = q["weight"] * np.exp(-q["delta_days"] / DECAY_DAYS)

    distinct_dates = q.groupby("customer_id")["response_date"].nunique()
    spans = (
        q.groupby("customer_id")["response_date"].max()
        - q.groupby("customer_id")["response_date"].min()
    ).dt.days
    earliest = q.groupby("customer_id")["response_date"].min()
    tenure_cutoff = latest - pd.Timedelta(days=30)
    stable_ids = set(
        distinct_dates[
            (distinct_dates >= 2)
            & (spans >= 14)
            & (earliest <= tenure_cutoff)
        ].index
    )
    q = q[q["customer_id"].isin(stable_ids)]

    rep_rows = []
    for cid, grp in q.groupby("customer_id"):
        scores = grp["score"].tolist()
        wts = grp["w_resp"].tolist()
        rep = weighted_median(scores, wts)
        if len(scores) >= 2 and all(int(s) == 6 for s in scores):
            rep = 8.0
        rep_rows.append((cid, rep))
    rep_df = pd.DataFrame(rep_rows, columns=["customer_id", "repr_score"])
    panel = complete.merge(rep_df, on="customer_id", how="inner").reset_index(drop=True)

    seg_targets = targets[targets["segment"] == segment]
    age_targets = (
        seg_targets[seg_targets["variable"] == "age_band"]
        .set_index("level")["target_proportion"]
        .to_dict()
    )
    region_targets = (
        seg_targets[seg_targets["variable"] == "region"]
        .set_index("level")["target_proportion"]
        .to_dict()
    )

    panel["w"] = 1.0
    n = len(panel)
    sweeps = 0
    for _ in range(100000):
        sweeps += 1
        before = panel["w"].copy()
        for level, share in age_targets.items():
            mask = panel["age_band"] == level
            current = panel.loc[mask, "w"].sum()
            if current > 0:
                panel.loc[mask, "w"] *= (share * n) / current
        for level, share in region_targets.items():
            mask = panel["region"] == level
            current = panel.loc[mask, "w"].sum()
            if current > 0:
                panel.loc[mask, "w"] *= (share * n) / current
        if (panel["w"] - before).abs().max() < CONVERGENCE:
            break

    cap_mask = panel["w"] > 1.5
    if cap_mask.any():
        amount_shaved = float((panel.loc[cap_mask, "w"] - 1.5).sum())
        uncapped_sum = float(panel.loc[~cap_mask, "w"].sum())
        if uncapped_sum > 0:
            panel.loc[~cap_mask, "w"] *= (uncapped_sum + amount_shaved) / uncapped_sum
        panel.loc[cap_mask, "w"] = 1.5

    total_w = panel["w"].sum()
    promoter_w = panel.loc[panel["repr_score"] >= 9, "w"].sum()
    detractor_w = panel.loc[panel["repr_score"] <= 6, "w"].sum()
    nps = round((promoter_w - detractor_w) / total_w * 100, 2) if total_w > 0 else 0.0
    return {
        "panel_size": n,
        "nps": float(nps),
        "promoter_count": int((panel["repr_score"] >= 9).sum()),
        "detractor_count": int((panel["repr_score"] <= 6).sum()),
        "sweeps": sweeps,
    }

In [ ]:
corp = segment_metrics("corp")
smb = segment_metrics("smb")

customer_nps_corp = corp["nps"]
customer_nps_smb = smb["nps"]
panel_size_corp = corp["panel_size"]
panel_size_smb = smb["panel_size"]
promoter_count_corp = corp["promoter_count"]
promoter_count_smb = smb["promoter_count"]
detractor_count_corp = corp["detractor_count"]
detractor_count_smb = smb["detractor_count"]

print(f"corp: panel={panel_size_corp}  P#={promoter_count_corp}  D#={detractor_count_corp}  NPS={customer_nps_corp}")
print(f"smb:  panel={panel_size_smb}   P#={promoter_count_smb}  D#={detractor_count_smb}  NPS={customer_nps_smb}")